# D30 Google Play (3.4M Apps and Games): feasibility check
Goal: decide whether the **games subset** can support a mobile module (label, pre-launch features, text, splits).
Run top to bottom. The last cell prints a summary to paste back. Nothing here trains a final model.

**Edit only `CSV_PATH` in the next cell.**

In [ ]:
import pandas as pd, numpy as np, json, warnings
warnings.filterwarnings("ignore")
pd.set_option("display.width",160); pd.set_option("display.max_columns",50)

CSV_PATH = "YOUR_FILE.csv"      # <-- path to the Kaggle csv (full file is fine)
MAX_GAMES_FOR_MODEL = 100_000   # cap for the quick model test
SUMMARY = {}

## 1. Load in chunks and keep games only (`genreId` starts with GAME)

In [ ]:
WANT = ["title","summary","genre","genreId","developer","developerId","free","price","offersIAP",
        "inAppProductPrice","adSupported","containsAds","len screenshots","releasedYear","releasedMonth",
        "releasedDayYear","score","ratings","reviews","minInstalls"]
header = pd.read_csv(CSV_PATH, nrows=0).columns.tolist()
print("Columns in file:", header)
missing = [c for c in WANT if c not in header]
print("Wanted but NOT in file:", missing)
use = [c for c in WANT if c in header]
parts=[]; total=0
for ch in pd.read_csv(CSV_PATH, usecols=use, chunksize=250_000, low_memory=False):
    total += len(ch)
    parts.append(ch[ch["genreId"].astype(str).str.startswith("GAME")])
g = pd.concat(parts, ignore_index=True)
print(f"Total rows: {total:,} | game rows: {len(g):,}")
g.to_csv("gp_games.csv", index=False)
SUMMARY.update(total_rows=total, game_rows=len(g), missing_columns=missing)

## 2. Genre coverage (is it more than one game type?)

In [ ]:
vc = g["genreId"].value_counts()
print(vc.to_string())
SUMMARY["n_game_genres"]=int(vc.size); SUMMARY["top_genre_share"]=round(float(vc.iloc[0]/len(g)),3)

## 3. Missing values in pre-launch fields

In [ ]:
print((g.isna().mean().round(3)*100).sort_values(ascending=False).to_string())
print("\nDuplicate titles:", g["title"].duplicated().sum(), "| duplicate summaries:", g["summary"].duplicated().sum())

## 4. Label feasibility\nMost apps are unrated, so we need a minimum ratings count. Table shows how many games survive each filter.

In [ ]:
g["score"]=pd.to_numeric(g["score"],errors="coerce"); g["ratings"]=pd.to_numeric(g["ratings"],errors="coerce")
print("Games with score > 0:", int((g["score"]>0).sum()))
rows=[]
for k in [1,10,25,50,100,500,1000]:
    s=g[(g["score"]>0)&(g["ratings"]>=k)]
    rows.append((k,len(s),round(s["score"].median(),2) if len(s) else None))
tab=pd.DataFrame(rows,columns=["min_ratings","games","median_score"]); print(tab.to_string(index=False))
SUMMARY["label_filter_table"]=tab.to_dict("records")

In [ ]:
MIN_R = 50   # change if the table above suggests otherwise
L = g[(g["score"]>0)&(g["ratings"]>=MIN_R)].copy()
print("Labelled games at MIN_R =",MIN_R,":",len(L))
print(L["score"].describe().round(2).to_string())
bal=[]
for t in [3.5,4.0,4.2,4.4,4.5]:
    bal.append((f"score>={t}", round(float((L["score"]>=t).mean()),3)))
med=L["score"].median(); bal.append((f"score>=median({med:.2f})",0.5))
print(pd.DataFrame(bal,columns=["rule","positive_share"]).to_string(index=False))
SUMMARY["labelled_games"]=len(L); SUMMARY["label_balance"]=bal; SUMMARY["median_score"]=float(med)

## 5. Time coverage (needed for temporal validation)

In [ ]:
print(L["releasedYear"].value_counts().sort_index().to_string())

## 6. Text: how much is there?

In [ ]:
for c in ["title","summary"]:
    ln=L[c].fillna("").str.len(); print(c,"length mean/median/p90:",round(ln.mean()),int(ln.median()),int(ln.quantile(.9)),"| empty:",int((ln==0).sum()))
SUMMARY["summary_len_mean"]=float(L["summary"].fillna("").str.len().mean())
print("\nSample rows:"); print(L[["title","summary","genreId","score","ratings"]].sample(8,random_state=1).to_string())

## 7. Developer concentration (affects how we split)

In [ ]:
dv=L["developer"].fillna("NA").value_counts()
print("Distinct developers:",dv.size,"| games per developer mean/median/max:",round(dv.mean(),2),int(dv.median()),int(dv.max()))
print("Share of games from developers with >=5 labelled games:",round(float(dv[dv>=5].sum()/len(L)),3))
print(dv.head(10).to_string())
SUMMARY.update(n_developers=int(dv.size), share_devs_ge5=round(float(dv[dv>=5].sum()/len(L)),3))

## 8. Leakage demo: how predictive are post-launch columns?\n(`minInstalls`, `ratings` must NOT be features. Seeing high AUC here confirms why.)

In [ ]:
from sklearn.metrics import roc_auc_score
L["y"]=(L["score"]>=L["score"].median()).astype(int)
for c in ["minInstalls","ratings","reviews"]:
    if c in L:
        x=pd.to_numeric(L[c],errors="coerce").fillna(0)
        print(c,"AUC vs label (rank-based):",round(roc_auc_score(L["y"],x),3))

## 9. Quick signal test (developer-grouped split, then temporal split)\nPre-launch features only. Compares structured-only vs structured + text.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

D = L.sample(min(len(L),MAX_GAMES_FOR_MODEL),random_state=0).reset_index(drop=True)
D["y"]=(D["score"]>=D["score"].median()).astype(int)
mm={m:i+1 for i,m in enumerate(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])}
D["releasedMonth"]=D["releasedMonth"].map(mm)
num=["price","len screenshots","releasedYear","releasedMonth"]
for c in ["free","offersIAP","adSupported","containsAds"]:
    if c in D: D[c]=D[c].astype(str).str.lower().isin(["1","true","1.0"]).astype(int); num.append(c)
for c in ["price","len screenshots"]:
    if c in D: D[c]=pd.to_numeric(D[c],errors="coerce")
num=[c for c in num if c in D]
D["genre_code"]=D["genreId"].astype("category").cat.codes; num.append("genre_code")
txt=(D["title"].fillna("")+". "+D["summary"].fillna(""))
def run(train,test,use_text):
    Xtr=D.loc[train,num].values; Xte=D.loc[test,num].values
    if use_text:
        tf=TfidfVectorizer(max_features=20000,min_df=3,stop_words="english").fit(txt[train])
        Z=tf.transform(txt[train]); sv=TruncatedSVD(max(2,min(50,Z.shape[1]-1)),random_state=0).fit(Z)
        Xtr=np.hstack([Xtr,sv.transform(tf.transform(txt[train]))]); Xte=np.hstack([Xte,sv.transform(tf.transform(txt[test]))])
    m=HistGradientBoostingClassifier(random_state=0).fit(Xtr,D.loc[train,"y"])
    p=m.predict_proba(Xte)[:,1]
    return f1_score(D.loc[test,"y"],(p>.5).astype(int),average="macro"), roc_auc_score(D.loc[test,"y"],p)
groups=D["developer"].fillna("NA")
res={}
for name,use_text in [("structured",False),("structured+text",True)]:
    f=[];a=[]
    for seed in range(3):
        tr,te=next(GroupShuffleSplit(1,test_size=.25,random_state=seed).split(D,groups=groups))
        m_,a_=run(tr,te,use_text); f.append(m_);a.append(a_)
    res[f"grouped_{name}"]=(round(np.mean(f),3),round(np.mean(a),3))
cut=D["releasedYear"].quantile(.75)
tr=D.index[D["releasedYear"]<=cut]; te=D.index[D["releasedYear"]>cut]
for name,use_text in [("structured",False),("structured+text",True)]:
    m_,a_=run(tr,te,use_text); res[f"temporal_{name}"]=(round(m_,3),round(a_,3))
print("Majority-class macro-F1 ~ 0.33 (balanced label). Results as (macro-F1, AUC):")
for k,v in res.items(): print(" ",k,v)
SUMMARY["quick_model"]={k:list(v) for k,v in res.items()}

## 10. Paste this back to Claude

In [ ]:
print(json.dumps(SUMMARY,indent=1,default=str))
json.dump(SUMMARY,open("gp_profile_summary.json","w"),indent=1,default=str)